<a href="https://colab.research.google.com/github/AliceFranca0/tcc-deteccao-fake-news-ptbr/blob/main/notebooks/Fase5_FineTuning_BERTimbau.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Sun Sep 13 21:11:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers datasets accelerate evaluate scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("/content/drive/MyDrive/TCC/fakerecogna_bruto.csv")
df = df.dropna(subset=["Noticia", "Classe"]).reset_index(drop=True)
df["Classe"] = df["Classe"].astype(int)

# MESMO random_state=42 dos baselines — essencial para a comparação ser justa
X_train, X_test, y_train, y_test = train_test_split(
    df["Noticia"], df["Classe"],
    test_size=0.2, random_state=42, stratify=df["Classe"]
)

print(f"Treino: {len(X_train)} | Teste: {len(X_test)}")

Treino: 9521 | Teste: 2381


In [ ]:
from transformers import AutoTokenizer
import datasets

MODELO = "neuralmind/bert-base-portuguese-cased"

# do_lower_case=False é obrigatório no BERTimbau — o modelo diferencia
# maiúsculas de minúsculas, e isso carrega informação em português
tokenizer = AutoTokenizer.from_pretrained(MODELO, do_lower_case=False)

def tokenizar(lote):
    return tokenizer(
        lote["texto"],
        truncation=True,      # corta textos longos demais
        padding="max_length", # iguala o tamanho de todos
        max_length=256        # 256 tokens cobre a maioria das notícias do corpus
    )

ds_train = datasets.Dataset.from_dict({"texto": X_train.tolist(), "labels": y_train.tolist()})
ds_test  = datasets.Dataset.from_dict({"texto": X_test.tolist(),  "labels": y_test.tolist()})

ds_train = ds_train.map(tokenizar, batched=True)
ds_test  = ds_test.map(tokenizar, batched=True)

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/9521 [00:00<?, ? examples/s]

Map:   0%|          | 0/2381 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

modelo = AutoModelForSequenceClassification.from_pretrained(MODELO, num_labels=2)

def calcular_metricas(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precisao, revocacao, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "acuracia": accuracy_score(labels, preds),
        "f1": f1,
        "precisao": precisao,
        "revocacao": revocacao,
    }

args = TrainingArguments(
    output_dir="/content/drive/MyDrive/TCC/bertimbau_resultados",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,          # acelera o treino na GPU T4
    logging_steps=50,
    report_to="none",   # evita pedir login no Weights & Biases
)

trainer = Trainer(
    model=modelo,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    compute_metrics=calcular_metricas,
)

trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Acuracia,F1,Precisao,Revocacao
1,0.123661,0.231130,0.939941,0.936416,0.994334,0.884874
2,0.083089,0.174020,0.960521,0.959095,0.994585,0.926050
3,0.019257,0.122149,0.976480,0.976291,0.983788,0.968908


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1788, training_loss=0.09562119254863236, metrics={'train_runtime': 506.0538, 'train_samples_per_second': 56.443, 'train_steps_per_second': 3.533, 'total_flos': 3757620537123840.0, 'train_loss': 0.09562119254863236, 'epoch': 3.0})

In [ ]:
resultados = trainer.evaluate()
print(resultados)

# Previsões para a matriz de confusão
previsoes = trainer.predict(ds_test)
y_pred_bert = previsoes.predictions.argmax(-1)

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred_bert, target_names=["Fake", "Real"]))
print(confusion_matrix(y_test, y_pred_bert))

# Salvar o modelo treinado (necessário para os Blocos B e C)
trainer.save_model("/content/drive/MyDrive/TCC/modelo_bertimbau_final")
tokenizer.save_pretrained("/content/drive/MyDrive/TCC/modelo_bertimbau_final")

Training Loss,Validation Loss,Epoch,Acuracia,F1,Precisao,Revocacao
0.019257,0.122149,3,0.976480,0.976291,0.983788,0.968908


{'eval_loss': 0.12214931845664978, 'eval_acuracia': 0.9764804703905922, 'eval_f1': 0.9762912785774767, 'eval_precisao': 0.9837883959044369, 'eval_revocacao': 0.9689075630252101}


              precision    recall  f1-score   support

        Fake       0.97      0.98      0.98      1191
        Real       0.98      0.97      0.98      1190

    accuracy                           0.98      2381
   macro avg       0.98      0.98      0.98      2381
weighted avg       0.98      0.98      0.98      2381

[[1172   19]
 [  37 1153]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/TCC/modelo_bertimbau_final/tokenizer_config.json',
 '/content/drive/MyDrive/TCC/modelo_bertimbau_final/tokenizer.json')

In [ ]:
# Exportar previsões para juntar com as dos baselines no Power BI
import pandas as pd

pd.DataFrame({
    "texto": X_test.values,
    "classe_real": y_test.values,
    "previsao_bertimbau": y_pred_bert,
}).to_csv("/content/drive/MyDrive/TCC/previsoes_bertimbau.csv",
          index=False, encoding="utf-8-sig")